# BEER

In [ ]:
import scipp as sc
import tof

import scippnexus as snx
from ess.reduce.unwrap import GenericUnwrapWorkflow
from ess.reduce.nexus.types import *
from ess.reduce.unwrap.types import *
from ess.reduce.unwrap.lut import LtotalRange, ChopperFrameSequence

## Mode 3 pulse shaping (MR) - PS2

### Chopper parameters

In [ ]:
choppers = {
    "PSC1": {
        "frequency": {"value": 168.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [144.0], "unit": "deg"},
        "distance": {"value": 6.45, "unit": "m"},
        "phase": {"value": 299.983856971683, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "PSC1",
    },
    "PSC2": {
        "frequency": {"value": 168.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [144.0], "unit": "deg"},
        "distance": {"value": 6.85, "unit": "m"},
        "phase": {"value": 299.983856971683, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "PSC2",
    },
    "FC1A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [72.0], "unit": "deg"},
        "distance": {"value": 8.283, "unit": "m"},
        "phase": {"value": -12.63240606395426, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC1A",
    },
    "FC2A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [175.0], "unit": "deg"},
        "distance": {"value": 79.975, "unit": "m"},
        "phase": {"value": 133.67285314925246, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC2A",
    },
}

In [ ]:
beer_choppers = {}
for key, ch in choppers.items():
    beer_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in beer_choppers.items():
    print(key)
    display(ch)

### Tof model

In [ ]:
source_position = sc.vector([0, 0, 0], unit="m")
source = tof.Source(facility="ess", neutrons=1_000_000, pulses=2)
detector = tof.Detector(distance=sc.scalar(158.0, unit="m"), name="detector")

params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in beer_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

### Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = beer_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = (
    sc.scalar(5.0, unit="m"),
    detector.distance - source_position.fields.z,
)
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot()

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()

## Mode 4 pulse shaping (HR) - PS3

### Chopper parameters

In [ ]:
choppers = {
    "PSC1": {
        "frequency": {"value": 168.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [144.0], "unit": "deg"},
        "distance": {"value": 6.45, "unit": "m"},
        "phase": {"value": 296.77336889692083, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "PSC1",
    },
    "PSC2": {
        "frequency": {"value": 168.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [144.0], "unit": "deg"},
        "distance": {"value": 6.65, "unit": "m"},
        "phase": {"value": 296.77336889692083, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "PSC2",
    },
    "FC1A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [72.0], "unit": "deg"},
        "distance": {"value": 8.283, "unit": "m"},
        "phase": {"value": -12.63240606395426, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC1A",
    },
    "FC2A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [175.0], "unit": "deg"},
        "distance": {"value": 79.975, "unit": "m"},
        "phase": {"value": 133.67285314925246, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC2A",
    },
}

In [ ]:
beer_choppers = {}
for key, ch in choppers.items():
    beer_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in beer_choppers.items():
    print(key)
    display(ch)

### Tof model

In [ ]:
params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in beer_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

### Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = beer_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = (
    sc.scalar(5.0, unit="m"),
    detector.distance - source_position.fields.z,
)
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot()

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()

## Mode 5 modulation (HF) 8X - M0+M1

### Chopper parameters

In [ ]:
choppers = {
    "FC1A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [72.0], "unit": "deg"},
        "distance": {"value": 8.283, "unit": "m"},
        "phase": {"value": -21.63240606395426, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC1A",
    },
    "MCA": {
        "frequency": {"value": 70.0, "unit": "Hz"},
        "open": {
            "value": [0.0, 45.0, 90.0, 135.0, 180.0, 225.0, 270.0, 315.0],
            "unit": "deg",
        },
        "close": {
            "value": [5.0, 50.0, 95.0, 140.0, 185.0, 230.0, 275.0, 320.0],
            "unit": "deg",
        },
        "distance": {"value": 9.3, "unit": "m"},
        "phase": {"value": 157.94241289703336, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "MCA",
    },
    "FC2A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [175.0], "unit": "deg"},
        "distance": {"value": 79.975, "unit": "m"},
        "phase": {"value": 133.67285314925246, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC2A",
    },
}

In [ ]:
beer_choppers = {}
for key, ch in choppers.items():
    beer_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in beer_choppers.items():
    print(key)
    display(ch)

### Tof model

In [ ]:
params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in beer_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

### Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = beer_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = (
    sc.scalar(5.0, unit="m"),
    detector.distance - source_position.fields.z,
)
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot()

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()

## Mode 10 modulation (MR) 16X - M2

### Chopper parameters

In [ ]:
choppers = {
    "FC1A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [72.0], "unit": "deg"},
        "distance": {"value": 8.283, "unit": "m"},
        "phase": {"value": -21.63240606395426, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC1A",
    },
    "MCB": {
        "frequency": {"value": 140.0, "unit": "Hz"},
        "open": {
            "value": [
                0.0,
                22.5,
                45.0,
                67.5,
                90.0,
                112.5,
                135.0,
                157.5,
                180.0,
                202.5,
                225.0,
                247.5,
                270.0,
                292.5,
                315.0,
                337.5,
            ],
            "unit": "deg",
        },
        "close": {
            "value": [
                5.0,
                27.5,
                50.0,
                72.5,
                95.0,
                117.5,
                140.0,
                162.5,
                185.0,
                207.5,
                230.0,
                252.5,
                275.0,
                297.5,
                320.0,
                342.5,
            ],
            "unit": "deg",
        },
        "distance": {"value": 9.35, "unit": "m"},
        "phase": {"value": 319.72252915855086, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "MCB",
    },
    "FC2A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [175.0], "unit": "deg"},
        "distance": {"value": 79.975, "unit": "m"},
        "phase": {"value": 133.67285314925246, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC2A",
    },
}

In [ ]:
beer_choppers = {}
for key, ch in choppers.items():
    beer_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in beer_choppers.items():
    print(key)
    display(ch)

### Tof model

In [ ]:
params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in beer_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

### Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = beer_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = (
    sc.scalar(5.0, unit="m"),
    detector.distance - source_position.fields.z,
)
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot()

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()

## Mode 11 modulation (HR) 16X - M3

### Chopper parameters

In [ ]:
choppers = {
    "FC1A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [72.0], "unit": "deg"},
        "distance": {"value": 8.283, "unit": "m"},
        "phase": {"value": -21.63240606395426, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC1A",
    },
    "MCB": {
        "frequency": {"value": 280.0, "unit": "Hz"},
        "open": {
            "value": [
                0.0,
                22.5,
                45.0,
                67.5,
                90.0,
                112.5,
                135.0,
                157.5,
                180.0,
                202.5,
                225.0,
                247.5,
                270.0,
                292.5,
                315.0,
                337.5,
            ],
            "unit": "deg",
        },
        "close": {
            "value": [
                5.0,
                27.5,
                50.0,
                72.5,
                95.0,
                117.5,
                140.0,
                162.5,
                185.0,
                207.5,
                230.0,
                252.5,
                275.0,
                297.5,
                320.0,
                342.5,
            ],
            "unit": "deg",
        },
        "distance": {"value": 9.35, "unit": "m"},
        "phase": {"value": 641.9450583171017, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "MCB",
    },
    "FC2A": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [0.0], "unit": "deg"},
        "close": {"value": [175.0], "unit": "deg"},
        "distance": {"value": 79.975, "unit": "m"},
        "phase": {"value": 133.67285314925246, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "FC2A",
    },
}

In [ ]:
beer_choppers = {}
for key, ch in choppers.items():
    beer_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in beer_choppers.items():
    print(key)
    display(ch)

### Tof model

In [ ]:
params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in beer_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

### Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = beer_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = (
    sc.scalar(5.0, unit="m"),
    detector.distance - source_position.fields.z,
)
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot()

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()